# Bundesliga predictor

In [2]:
import pandas as pd 
import requests
import numpy as np
from collections import defaultdict
from typing import Dict

In [3]:
def get_table(season: int) -> pd.DataFrame:

    url = f"https://api.openligadb.de/getbltable/bl1/{season}"
    data = requests.get(url).json()

    data

    rows = []
    for team in data:
        rows.append({
            "team_name" : team["teamName"],
            "points" : team["points"],
            "win" : team["won"],
            "draw" : team["draw"],
            "loss" : team["lost"],
            "goals_for" : team["goals"],
            "goals_against" : team["opponentGoals"],
            "goal_diff" : team["goalDiff"]
            })
    return pd.DataFrame(rows)

In [9]:
get_table(2023)

,team_name,points,win,draw,loss,goals_for,goals_against,goal_diff
0,Bayer 04 Leverkusen,90,28,6,0,89,24,65
1,VfB Stuttgart,73,23,4,7,78,39,39
2,FC Bayern München,72,23,3,8,94,45,49
3,RB Leipzig,65,19,8,7,77,39,38
4,Borussia Dortmund,63,18,9,7,68,43,25
5,Eintracht Frankfurt,47,11,14,9,51,50,1
6,TSG Hoffenheim,46,13,7,14,66,66,0
7,1. FC Heidenheim 1846,42,10,12,12,50,55,-5
8,SV Werder Bremen,42,11,9,14,48,54,-6
9,SC Freiburg,42,11,9,14,45,58,-13


In [3]:
def get_matchday(season: int, matchday: int) -> pd.DataFrame:
    url = f"https://api.openligadb.de/getmatchdata/bl1/{season}/{matchday}"
    data = requests.get(url).json()
    
    rows = []
    for match in data:
        rows.append({
            "matchday": matchday,
            "date": match["matchDateTime"],
            "home": match["team1"]["teamName"],
            "away": match["team2"]["teamName"],
            "score_home": match["matchResults"][-1]["pointsTeam1"] if match["matchResults"] else None,
            "score_away": match["matchResults"][-1]["pointsTeam2"] if match["matchResults"] else None,
        })
    return pd.DataFrame(rows)

# Pull a full season
all_matches = pd.concat([get_matchday(2023, md) for md in range(1, 35)])

In [5]:
all_matches

,matchday,date,home,away,score_home,score_away
0,1,2023-08-18T20:30:00,SV Werder Bremen,FC Bayern München,0,4
1,1,2023-08-19T15:30:00,Bayer 04 Leverkusen,RB Leipzig,3,2
2,1,2023-08-19T15:30:00,VfL Wolfsburg,1. FC Heidenheim 1846,2,0
3,1,2023-08-19T15:30:00,TSG Hoffenheim,SC Freiburg,1,2
4,1,2023-08-19T15:30:00,FC Augsburg,Borussia Mönchengladbach,4,4
...,...,...,...,...,...,...
4,34,2024-05-18T15:30:00,VfL Wolfsburg,1. FSV Mainz 05,1,3
5,34,2024-05-18T15:30:00,TSG Hoffenheim,FC Bayern München,4,2
6,34,2024-05-18T15:30:00,SV Werder Bremen,VfL Bochum,4,1
7,34,2024-05-18T15:30:00,VfB Stuttgart,Borussia Mönchengladbach,4,0


In [ ]:
def create_table(season: int) -> pd.DataFrame:

    season_matches = pd.concat([get_matchday(season, md) for md in range(1, 35)])

    teams: Dict[str, Dict[str, int]] = defaultdict(lambda: {
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "home_wins": 0,
            "home_losses": 0
        })

    for _, row in season_matches.iterrows():
        home = row["home"]
        away = row["away"]

        home_goals = row["score_home"]
        away_goals = row["score_away"]

        teams[home]["goals_for"] += home_goals
        teams[home]["goals_against"] += away_goals
        teams[away]["goals_for"] += away_goals
        teams[away]["goals_against"] += home_goals

        if home_goals > away_goals:
            teams[home]["wins"] += 1
            teams[home]["home_wins"] += 1
            teams[away]["losses"] += 1
        elif home_goals < away_goals:
            teams[away]["wins"] += 1
            teams[home]["home_losses"] += 1
            teams[home]["losses"] += 1
        else:
            teams[away]["draws"] += 1
            teams[home]["draws"] += 1

    data = []

    for team, statistics in teams.items():
        goal_diff = statistics["goals_for"] - statistics["goals_against"]
        points = statistics["wins"] * 3 + statistics["draws"] 

        data.append({
            "team_name" : team,
            "points" : points,
            "win" : statistics["wins"],
            "draw" : statistics["draws"],
            "loss" : statistics["losses"],
            "goals_for" : statistics["goals_for"],
            "goals_against" : statistics["goals_against"],
            "goal_diff" : goal_diff,
            "home_wins" : statistics["home_wins"],
            "home_losses" : statistics["home_losses"]
        })

    table = pd.DataFrame(data)

    table = table.sort_values(["points", "goal_diff", "goals_for"], ascending= [False, False, False]).reset_index(drop= True)

    table["position"] = table.index + 1

    return table

In [5]:
create_table(2024)

,team_name,points,win,draw,loss,goals_for,goals_against,goal_diff,home_wins,home_losses,position
0,FC Bayern München,82,25,7,2,99,32,67,14,1,1
1,Bayer 04 Leverkusen,69,19,12,3,72,43,29,10,3,2
2,Eintracht Frankfurt,60,17,9,8,68,46,22,10,3,3
3,Borussia Dortmund,57,17,6,11,71,51,20,11,3,4
4,SC Freiburg,55,16,7,11,49,53,-4,9,5,5
5,1. FSV Mainz 05,52,14,10,10,55,43,12,6,3,6
6,RB Leipzig,51,13,12,9,53,48,5,8,3,7
7,SV Werder Bremen,51,14,9,11,54,57,-3,5,6,8
8,VfB Stuttgart,50,14,8,12,64,53,11,7,8,9
9,Borussia Mönchengladbach,45,13,6,15,55,57,-2,7,7,10
